In [108]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("train.csv").set_index("Id")
df

,PassengerId,TicketClass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilyOnboard,Survived
Id,,,,,,,,,,
1,1,3,female,11.7,1,0,14.45,Batavia,no,0
2,2,2,female,38.9,0,0,51.96,Semarang,no,1
3,3,3,female,9.6,0,0,10.98,Batavia,yes,0
4,4,3,male,21.6,0,0,4.79,Batavia,yes,0
5,5,1,male,27.0,2,0,50.50,Semarang,no,0
...,...,...,...,...,...,...,...,...,...,...
496,496,3,male,18.6,1,0,8.15,Semarang,no,0
497,497,2,female,23.2,1,1,NaN,Surabaya,yes,1
498,498,3,male,31.5,1,0,12.53,Semarang,yes,0


## Ver 1: impute with mean and mode (baru sadar ini data leakage)

In [131]:
df = pd.read_csv("train.csv").set_index("Id")
print(df.isna().sum())

df["Age"] = df["Age"].fillna(df["Age"].mean())
df["Fare"] = df["Fare"].fillna(df["Fare"].mean())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print(df.isna().sum())

PassengerId       0
TicketClass       0
Sex               0
Age              22
SibSp             0
Parch             0
Fare             20
Embarked         21
FamilyOnboard     0
Survived          0
dtype: int64
PassengerId      0
TicketClass      0
Sex              0
Age              0
SibSp            0
Parch            0
Fare             0
Embarked         0
FamilyOnboard    0
Survived         0
dtype: int64


## Ver 2: dropna

In [150]:
df = pd.read_csv("train.csv").set_index("Id")
print(df.isna().sum())

df = df.dropna()

print(df.isna().sum())

PassengerId       0
TicketClass       0
Sex               0
Age              22
SibSp             0
Parch             0
Fare             20
Embarked         21
FamilyOnboard     0
Survived          0
dtype: int64
PassengerId      0
TicketClass      0
Sex              0
Age              0
SibSp            0
Parch            0
Fare             0
Embarked         0
FamilyOnboard    0
Survived         0
dtype: int64


## Data encoding

In [151]:
from sklearn.model_selection import train_test_split

xall, yall = df.iloc[:, :-1].to_numpy(), df.iloc[:, -1].to_numpy()

x, xt, y, yt = train_test_split(xall, yall, test_size=0.2, random_state=67)
print(x.shape)

(351, 9)


In [153]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

ct = ColumnTransformer([
    ("oe", OrdinalEncoder(categories=[["no", "yes"]]), [8]),
    ("ohe", OneHotEncoder(), [2, 7])
], remainder="passthrough")

## Model processing

In [338]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(
    ct,
    StandardScaler(),
    PowerTransformer(),
    StandardScaler(),
    PolynomialFeatures(degree=5),
    StandardScaler(),
    VarianceThreshold(),
    SelectKBest(score_func=f_classif),
    # KNeighborsClassifier(),
    # SVC(kernel="rbf", C=3593.813663804626, gamma=0.31622776601683794),
    # GradientBoostingClassifier(),
    LogisticRegression()
)
skf = StratifiedKFold(n_splits=8, shuffle=True, random_state=67)

gscv = GridSearchCV(
    estimator=pipe,
    param_grid={
        # "kneighborsclassifier__n_neighbors": [12],
        # "svc__C": [3593.813663804626],
        # "svc__gamma": [0.31622776601683794],
        "selectkbest__k": [3],
        # "gradientboostingclassifier__n_estimators": [10],
        # "gradientboostingclassifier__max_depth": [1],
        "logisticregression__C": np.logspace(-3.5, -2.5, 5),
    },
    cv=skf
)
gscv.fit(x, y)
bestmodel = gscv.best_estimator_

print(gscv.best_params_)
gscv.best_score_

{'logisticregression__C': np.float64(0.0005623413251903491), 'selectkbest__k': 3}


np.float64(0.7151823467230445)

## Accuracy test

In [339]:
from sklearn.metrics import accuracy_score

accuracy_score(yt, bestmodel.predict(xt))

0.6477272727272727

## Refit all

In [340]:
gscv.fit(xall, yall)
bestmodel = gscv.best_estimator_

## Submission

In [342]:
dfx = pd.read_csv("test.csv").set_index("Id")

print(dfx.isna().sum())
dfx["Age"] = dfx["Age"].fillna(dfx["Age"].mean())
dfx["Fare"] = dfx["Fare"].fillna(dfx["Fare"].mean())
dfx["Embarked"] = dfx["Embarked"].fillna(dfx["Embarked"].mode()[0])
print(dfx.isna().sum())

PassengerId       0
TicketClass       0
Sex               0
Age              15
SibSp             0
Parch             0
Fare             15
Embarked          8
FamilyOnboard     0
dtype: int64
PassengerId      0
TicketClass      0
Sex              0
Age              0
SibSp            0
Parch            0
Fare             0
Embarked         0
FamilyOnboard    0
dtype: int64


In [343]:
xsub = dfx.to_numpy()
ysub = bestmodel.predict(xsub)

# flipidx = np.linspace(3, 499, 150, dtype=int)
# ysub[flipidx] = 1 - ysub[flipidx]

sub = pd.Series(ysub, index=dfx.index, name="Survived")
sub.to_csv("submission.csv")

sub

Id
501     0
502     0
503     0
504     0
505     0
       ..
996     0
997     1
998     0
999     0
1000    0
Name: Survived, Length: 500, dtype: int64